In [0]:
import os, sys
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, repo_root)

from src import config_loader as cl
from src import incremental as inc
from src import steps

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("resource", "ALL", "Resource (ALL or e.g. Patient)")
dbutils.widgets.text("start_date", "", "Force start date (blank = watermark)")
dbutils.widgets.text("end_date", "", "End date (blank = last complete day)")
dbutils.widgets.text("max_days", "", "Max days per run (blank = config)")
dbutils.widgets.text("max_pages", "", "Max pages override (blank = config)")
dbutils.widgets.text("catalog", "", "Catalog override (optional)")
dbutils.widgets.text("run_id", "", "Run id (blank = generate)")
dbutils.widgets.text("pipeline_name", "manual_incremental", "Pipeline name")

g = lambda k: dbutils.widgets.get(k).strip()
resource_param = g("resource")
start_override, end_override = g("start_date") or None, g("end_date") or None
max_days = int(g("max_days")) if g("max_days") else None
max_pages = int(g("max_pages")) if g("max_pages") else None
run_id = g("run_id") or cl.new_batch_id()
pipeline_name = g("pipeline_name")

overrides = {"project": {"catalog": g("catalog")}} if g("catalog") else None
cfg = cl.load_config(os.path.join(repo_root, "config", "config.yaml"), overrides)
fmt = cfg["api"]["format"]

only = None if resource_param.upper() == "ALL" else [resource_param]
resources = cl.get_resources(cfg, only)      # Patient -> Encounter -> Observation -> Condition

print(f"run_id={run_id} | now (UTC)={cl.utc_now():%Y-%m-%d %H:%M}")
print(f"latest complete UTC day = {inc.latest_complete_day(cfg)}")

In [0]:
from src import meta
meta.create_meta_tables(spark, cfg)

plans = {}
for r in resources:
    last = inc.get_last_load_date(spark, cfg, r["name"])
    p = inc.plan_load_dates(cfg, last, start_override, end_override, max_days)
    p["last_load_date"] = last
    plans[r["name"]] = p

display(pd.DataFrame([{
    "resource": n, "last_load_date": p["last_load_date"], "start": p["start"],
    "start_from": p["start_source"], "end": p["end"],
    "days_to_load": ", ".join(p["dates"]) or "(nothing: up to date)",
} for n, p in plans.items()]))

In [0]:
results = []
for r in resources:
    name = r["name"]
    dates = plans[name]["dates"]
    if not dates:
        results.append({"resource": name, "load_date": None, "status": "UP_TO_DATE",
                        "raw_records": None, "bronze_rows": None, "detail": "nothing new"})
        continue

    for d in dates:
        raw = steps.run_raw_day(spark, cfg, name, d, max_pages, fmt, run_id, pipeline_name)
        if raw["status"] != "SUCCESS":
            results.append({"resource": name, "load_date": d, "status": "RAW_FAILED",
                            "raw_records": raw["records_fetched"], "bronze_rows": None,
                            "detail": raw["error"]})
            break
        if raw["truncated"]:
            results.append({"resource": name, "load_date": d, "status": "TRUNCATED",
                            "raw_records": raw["records_fetched"], "bronze_rows": None,
                            "detail": "page cap reached; rerun with a higher max_pages"})
            break

        b = steps.run_bronze_day(spark, cfg, name, d, run_id, pipeline_name, fmt)
        if b["status"] != "SUCCESS":
            results.append({"resource": name, "load_date": d, "status": "BRONZE_FAILED",
                            "raw_records": raw["records_fetched"], "bronze_rows": b["bronze_rows"],
                            "detail": b["error"] or b["status"]})
            break

        inc.update_watermark(spark, cfg, name, d, b["batch_id"])      # only after full success
        results.append({"resource": name, "load_date": d, "status": "LOADED",
                        "raw_records": raw["records_fetched"], "bronze_rows": b["bronze_rows"],
                        "detail": ""})
        print(f"{name} {d}: loaded {b['bronze_rows']} rows, watermark -> {d}")

res_df = pd.DataFrame(results)
display(res_df)

problems = res_df[~res_df["status"].isin(["LOADED", "UP_TO_DATE"])]
if len(problems):
    raise Exception(f"Incremental load problems:\n{problems.to_string()}")

In [0]:
display(spark.sql(f"""
  SELECT resource_type, last_load_date, watermark_ts, last_batch_id, last_status, updated_at
  FROM {cl.meta_table(cfg, 'watermark')} ORDER BY resource_type
"""))